In [ ]:
!nvidia-smi

In [ ]:
# Install vLLM + utilities
uv pip install -q vllm lm-format-enforcer pandas datasets scikit-learn

In [ ]:
import os
import re
import json
import time
import numpy as np
from datasets import load_dataset
from tqdm import tqdm

print("Loading SuperGLUE BoolQ...")

# Use validation split (has labels); test split often has no labels in HF
ds = load_dataset("super_glue", "boolq", split="validation")

print(f"Dataset size: {len(ds)} examples")
print(f"Columns: {ds.column_names}")
# answer: True/False
print(f"Label distribution: {dict(zip(*np.unique([str(x) for x in ds['answer']], return_counts=True)))}")

In [ ]:
# --- Config ---
MODEL_NAME = "Qwen/Qwen3-8B"
MAX_NEW_TOKENS = 32
EVAL_SIZE = len(ds)   # or e.g. 100 for a quick smoke test
ENABLE_THINKING = False

SYSTEM_PROMPT = (
    "You are a yes/no question answering system. "
    "Given a passage and a question, answer with exactly one word: Yes or No."
)

print({
    "model": MODEL_NAME,
    "eval_size": EVAL_SIZE,
    "max_new_tokens": MAX_NEW_TOKENS,
    "enable_thinking": ENABLE_THINKING,
})

In [ ]:
if __name__ == '__main__':
    from transformers import AutoTokenizer
    from vllm import LLM, SamplingParams

    print("Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

    print("Loading Qwen3-8B with vLLM...")
    llm = LLM(
        model=MODEL_NAME,
        dtype="float16",
        gpu_memory_utilization=0.95,
        max_model_len=4096,
        enforce_eager=False,
    )

    sampling_params = SamplingParams(
        temperature=0,
        top_k=20,
        max_tokens=MAX_NEW_TOKENS,
    )

    print("Model loaded successfully!")

In [ ]:
# --- Output directory (Colab Drive if available) ---
IN_COLAB = False
try:
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")
    DRIVE_SAVE_DIR = "/content/drive/MyDrive/Colab_Data/Qwen3_8B_BoolQ_Eval_vLLM"
else:
    DRIVE_SAVE_DIR = os.path.abspath("./qwen3_8b_boolq_eval_vllm_outputs")

os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)
print(f"Saving results to: {DRIVE_SAVE_DIR}")

In [ ]:
# --- Checkpoint setup ---
CHECKPOINT_FILE = os.path.join(DRIVE_SAVE_DIR, "qwen3_8b_boolq_vllm_checkpoint.jsonl")
OUTPUTS_CACHE   = os.path.join(DRIVE_SAVE_DIR, "qwen3_8b_boolq_vllm_raw_outputs.json")

results = []
if os.path.exists(CHECKPOINT_FILE):
    print(f"Found checkpoint: {CHECKPOINT_FILE}")
    with open(CHECKPOINT_FILE) as f:
        for line in f:
            try:
                results.append(json.loads(line))
            except json.JSONDecodeError:
                continue
    print(f"Loaded {len(results)} previously evaluated examples.")
else:
    print("No checkpoint found — starting from scratch.")

passages  = ds["passage"]
questions = ds["question"]
answers   = ds["answer"]   # True / False

already_done = len(results)
remaining_passages  = passages[already_done:EVAL_SIZE]
remaining_questions = questions[already_done:EVAL_SIZE]
remaining_answers   = answers[already_done:EVAL_SIZE]

def build_prompt(passage: str, question: str) -> str:
    user_content = f"Passage:\n{passage}\n\nQuestion: {question}\n\nAnswer (Yes or No):"
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_content},
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=ENABLE_THINKING,
    )

prompts = [
    build_prompt(p, q) for p, q in zip(remaining_passages, remaining_questions)
]
print(f"Built {len(prompts)} prompts (enable_thinking={ENABLE_THINKING}).")
print("\nExample prompt (first 500 chars):")
print(prompts[0][:500] if prompts else "(none — all done)")

In [ ]:
# --- Generate with vLLM ---
if prompts:
    print(f"Generating responses for {len(prompts)} prompts...")

    gen_start = time.time()
    outputs = llm.generate(prompts, sampling_params)
    gen_time = time.time() - gen_start

    total_new_tokens = sum(len(o.outputs[0].token_ids) for o in outputs)
    throughput = total_new_tokens / gen_time if gen_time > 0 else None

    print("\nGeneration complete.")
    print(f"  Time:       {gen_time/60:.1f} min")
    print(f"  Tokens:     {total_new_tokens:,}")
    print(f"  Throughput: {throughput:.1f} tokens/sec" if throughput else "  Throughput: N/A")

    raw_cache = [
        {
            "idx": already_done + i,
            "passage": remaining_passages[i],
            "question": remaining_questions[i],
            "ground_truth": remaining_answers[i],
            "response": o.outputs[0].text,
            "n_tokens": len(o.outputs[0].token_ids),
        }
        for i, o in enumerate(outputs)
    ]

    with open(OUTPUTS_CACHE, "w") as f:
        json.dump(raw_cache, f)
    print(f"Raw outputs cached to: {OUTPUTS_CACHE}")
else:
    print("No prompts to generate — loading cached outputs (if present).")
    if not os.path.exists(OUTPUTS_CACHE):
        raw_cache = []
    else:
        with open(OUTPUTS_CACHE) as f:
            raw_cache = json.load(f)
    gen_time = None
    throughput = None
    total_new_tokens = sum(r.get("n_tokens", 0) for r in raw_cache)

print(f"Raw items available for scoring: {len(raw_cache)}")

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

def strip_thinking(text: str) -> str:
    if "</think>" in text:
        return text.split("</think>", 1)[-1].strip()
    return text

def extract_bool(text: str):
    """Map model output to True/False. Returns None if unparseable."""
    text = strip_thinking(text).strip().lower()
    if "yes" in text and "no" not in text[: text.find("yes") + 3]:
        return True
    if text.startswith("yes"):
        return True
    if "no" in text and "yes" not in text[: text.find("no") + 2]:
        return False
    if text.startswith("no"):
        return False
    if "true" in text:
        return True
    if "false" in text:
        return False
    return None

# --- Score new outputs and append to checkpoint ---
new_results = []
for item in raw_cache:
    idx = item["idx"]
    response = item["response"]
    pred = extract_bool(response)
    gt = item["ground_truth"]

    res = {
        "idx": idx,
        "ground_truth": gt,
        "prediction": pred,
        "raw_response": response.strip(),
        "is_correct": (pred == gt) if pred is not None else False,
        "is_unknown": (pred is None),
        "new_tokens": int(item.get("n_tokens", 0)),
    }
    new_results.append(res)

if new_results:
    with open(CHECKPOINT_FILE, "a") as f:
        for r in new_results:
            f.write(json.dumps(r) + "\n")
    results.extend(new_results)

print(f"Total scored results: {len(results)}")

# --- Final metrics ---
all_gt = [r["ground_truth"] for r in results]
all_pred = [r["prediction"] if r["prediction"] is not None else "unknown" for r in results]
# For sklearn we need a label; treat unknown as wrong for accuracy
pred_for_acc = [r["prediction"] for r in results if r["prediction"] is not None]
gt_for_acc = [r["ground_truth"] for r in results if r["prediction"] is not None]

correct_count = sum(1 for r in results if r["is_correct"])
unknown_count = sum(1 for r in results if r["is_unknown"])
all_new_tokens = sum(r["new_tokens"] for r in results)
accuracy = correct_count / len(results) if results else 0
accuracy_known = accuracy_score(gt_for_acc, pred_for_acc) if pred_for_acc else 0

final_metrics = {
    "method": f"0_shot_vllm_thinking={ENABLE_THINKING}",
    "model": MODEL_NAME,
    "dataset": "super_glue/boolq:validation",
    "eval_size": len(results),
    "accuracy": accuracy,
    "accuracy_known_only": accuracy_known if pred_for_acc else "N/A",
    "unknown_frac": unknown_count / len(results) if results else 0,
    "total_new_tokens": all_new_tokens,
    "gen_tokens_per_sec": throughput if throughput is not None else "N/A (loaded from cache)",
    "total_gen_time_min": (gen_time / 60) if gen_time is not None else "N/A (loaded from cache)",
}

print("\n=== Final Metrics ===")
for k, v in final_metrics.items():
    print(f"  {k}: {v}")

print("\n=== Classification Report (True/False) ===")
labels = [True, False]
pred_labels = [p for p in all_pred if p != "unknown"]
gt_labels = [all_gt[i] for i in range(len(all_gt)) if all_pred[i] != "unknown"]
if pred_labels:
    print(classification_report(gt_labels, pred_labels, labels=labels))
    print("Confusion matrix (rows=true, cols=pred):")
    print(confusion_matrix(gt_labels, pred_labels, labels=labels))

METRICS_FILE = os.path.join(DRIVE_SAVE_DIR, "final_metrics.json")
with open(METRICS_FILE, "w") as f:
    json.dump(final_metrics, f, indent=2)
print(f"\nMetrics saved to: {METRICS_FILE}")